In [ ]:
import json
import math
from dataclasses import replace

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from uavloc.data.rasters import load_raster
from uavloc.features.factory import build_features, correlator_config
from uavloc.filtering.hypotheses import HypothesisTracker, HypothesisTrackerConfig
from uavloc.filtering.particles import ParticleFilter, ParticleFilterConfig
from uavloc.geo import LatLon, bearing_deg, east_north_m, offset_latlon
from uavloc.matching.correlator import CAMERA_ASPECT_W_OVER_H, CorrelatorConfig, PositionCorrelator
from uavloc.matching.window import SearchWindow, score_window
from uavloc.simulation.camera import CameraConfig, RasterCamera
from uavloc.simulation.correctors import InsOnly
from uavloc.simulation.estimators import InsOnlyEstimator, ReplaceEstimator, SimulatedFrames
from uavloc.simulation.flight import FlightConfig, simulate_flight
from uavloc.simulation.ins import InsConfig, InsSimulator
from uavloc.simulation.route import Pose, sample_route
from uavloc.simulation.sequence import fly, load_flight_set
from uavloc.simulation.synthetic import ColourClassFeatures, class_world, paint
from uavloc.track import REPO_ROOT, set_seed

# Params

In [ ]:
SEED = 42
DEVICE = "cpu"
ROUTE_SET = "sim-flights"
REFERENCE_RUN = "tools/flight_simulator/outputs/bushcha-loop"   # the build's flight that the replace arm must reproduce
CORRELATOR = {
    "method": "segmentation", "seg_architecture": "lraspp",
    "seg_checkpoint_drone": "pocs/mobilenet-lraspp-speed/model-lraspp-bn-satellite.pt",
    "seg_checkpoint_sat": "pocs/mobilenet-lraspp-speed/model-lraspp-bn-satellite.pt",
    "seg_channels": "all", "drone_seg_gsd_m": 0.385, "match_gsd_m": 0.385, "matching": "weighted-ncc",
    "class_weights": [0.0, 1.0, 1.0, 0.5, 0.0, 0.5], "window": "rect", "search_scale": 1.7, "fov_w_deg": 60.0,
    "sat_feature_crop_pct": 0.03, "drone_feature_crop_pct": 0.0,
}
CHECK_VILLAGE_ROUTES = 2           # tuning routes for the INS-only and window checks
SELF_MATCH_INITIAL_ERROR_M = 10.0
SELF_MATCH_FIX = 3                 # converged within 1 m by this fix
CONVERGED_M = 1.0
SYNTHETIC_MPP = 0.385
SYNTHETIC_SIZE_M = 400.0
ROAD_WIDTH_M = 6.0
BLOCK_SIZE_M = 15.0
BLOCK_SEPARATION_M = 60.0
DISTINGUISH_ROAD_NORTH_M = 50.0
ELONGATION_MIN = 3.0
TWIN_WEIGHT_RATIO_MIN = 0.9
PRUNED_WEIGHT_MAX = 0.01
MAX_FIX_MS = 1000.0
ALTITUDE_M = 40.0
SPEED_MPS = 10.0
STEP_S = 0.5
FIX_INTERVAL_S = 5.0

In [ ]:
set_seed(SEED)
checks = {}

# Data

In [ ]:
set_config, tuning_specs = load_flight_set(ROUTE_SET, split="tuning")
reference = json.loads((REPO_ROOT / REFERENCE_RUN / "run.json").read_text())
reference_track = pd.read_csv(REPO_ROOT / REFERENCE_RUN / "track.csv")
specs = tuning_specs[:CHECK_VILLAGE_ROUTES]
[(s.village, s.route_index) for s in specs]

In [ ]:
def build_correlator(map_raster) -> PositionCorrelator:
    features = build_features(CORRELATOR, map_raster, DEVICE)
    return PositionCorrelator(features, replace(correlator_config(CORRELATOR), aspect_w_over_h=CAMERA_ASPECT_W_OVER_H))

In [ ]:
def synthetic_correlator(probabilities) -> PositionCorrelator:
    return PositionCorrelator(ColourClassFeatures(probabilities, CORRELATOR["match_gsd_m"]),
                              replace(correlator_config(CORRELATOR), aspect_w_over_h=CAMERA_ASPECT_W_OVER_H))

In [ ]:
def straight_poses(start: LatLon, bearing: float, length_m: float) -> list[Pose]:
    end = offset_latlon(start, length_m * math.sin(math.radians(bearing)), length_m * math.cos(math.radians(bearing)))
    return sample_route([start, end], SPEED_MPS, STEP_S, ALTITUDE_M)

In [ ]:
def long_axis(cov: np.ndarray) -> tuple[float, float]:
    """`(s1 / s2, compass bearing of the long axis in [0, 180))`."""
    values, vectors = np.linalg.eigh(cov)
    return float(np.sqrt(values[1] / values[0])), bearing_deg(*vectors[:, 1]) % 180.0

In [ ]:
def make_filters(correlator) -> dict:
    return {"particles": ParticleFilter(correlator, ParticleFilterConfig(), SEED),
            "hypotheses": HypothesisTracker(correlator, HypothesisTrackerConfig())}

# Run

In [ ]:
ref_drone, ref_map = load_raster(reference["drone_source"]), load_raster(reference["map_source"])
ref_correlator = build_correlator(ref_map)
ref_poses = sample_route([LatLon(*w) for w in reference["waypoints"]], reference["speed_mps"], reference["step_s"],
                         reference["altitude_m"])
ref_camera = RasterCamera(ref_drone, CameraConfig(**reference["camera"]))
track, _ = fly(ref_poses, ref_camera, InsSimulator(InsConfig(**reference["ins"]), reference["seed"]),
               ReplaceEstimator(ref_correlator), reference["correction_interval_s"], 1.0)
difference = float(np.abs(track["est_error_m"].to_numpy() - reference_track["est_error_m"].to_numpy()).max())
checks["replace_reproduces_simulator"] = {"end_error_m": float(track["est_error_m"].iloc[-1]),
                                          "max_abs_diff_m": difference, "pass": difference < 1e-9}
checks["replace_reproduces_simulator"]

In [ ]:
village_correlators, results = {}, []
for spec in specs:
    drone_raster = load_raster({"kind": "multidate", "place": spec.village, "date": set_config.drone_date})
    village_correlators[spec.village] = build_correlator(
        load_raster({"kind": "multidate", "place": spec.village, "date": set_config.map_date}))
    poses = sample_route(spec.waypoints, set_config.speed_mps, set_config.step_s, set_config.altitude_m)
    camera = RasterCamera(drone_raster, CameraConfig(set_config.fov_w_deg))
    ours, _ = fly(poses, camera, InsSimulator(set_config.ins, spec.seed), InsOnlyEstimator(), set_config.correction_interval_s, 1.0)
    theirs = simulate_flight(poses, camera, InsSimulator(set_config.ins, spec.seed), InsOnly(),
                             FlightConfig(set_config.correction_interval_s)).track
    results.append(max(float(np.abs(ours["ins_error_m"] - theirs["ins_error_m"]).max()),
                       float(np.abs(ours["est_error_m"] - ours["ins_error_m"]).max())))
checks["ins_only_matches_simulator"] = {"max_abs_diff_m": max(results), "pass": max(results) == 0.0}
checks["ins_only_matches_simulator"]

In [ ]:
frames, rows = SimulatedFrames(), []
for spec in specs:
    correlator = village_correlators[spec.village]
    drone_raster = load_raster({"kind": "multidate", "place": spec.village, "date": set_config.drone_date})
    camera = RasterCamera(drone_raster, CameraConfig(set_config.fov_w_deg))
    poses = sample_route(spec.waypoints, set_config.speed_mps, set_config.step_s, set_config.altitude_m)
    for pose in poses[::int(FIX_INTERVAL_S / STEP_S)][1:]:
        frame = frames.frame(camera.view(pose), pose.position, pose.yaw_deg, pose.altitude_m)
        old = correlator.surface(frame, pose.position)
        window = SearchWindow(pose.position, pose.yaw_deg, correlator.search_half_range_m(frame))
        new = score_window(correlator, correlator.drone_map(frame), correlator.footprint_m(frame), window)
        correlator.clear_cache()
        same_shape = old.heatmap.shape == new.heatmap.shape
        rows.append({"same_shape": same_shape,
                     "same_argmax": same_shape and np.argmax(old.heatmap) == np.argmax(new.heatmap),
                     "max_abs_dz": float(np.abs(old.heatmap - new.heatmap).max()) if same_shape else np.inf,
                     "blind": bool(np.ptp(old.heatmap) == 0)})
window_rows = pd.DataFrame(rows)
content = window_rows[~window_rows["blind"]]
checks["window_reproduces_surface"] = {
    "frames": len(window_rows), "content_frames": len(content), "same_shape": bool(window_rows["same_shape"].all()),
    "same_argmax_share": float(content["same_argmax"].mean()), "max_abs_dz": float(window_rows["max_abs_dz"].max()),
    "pass": bool(window_rows["same_shape"].all() and content["same_argmax"].all() and window_rows["max_abs_dz"].max() < 1e-3)}
checks["window_reproduces_surface"]

In [ ]:
spec = specs[0]
drone_raster = load_raster({"kind": "multidate", "place": spec.village, "date": set_config.drone_date})
self_correlator = build_correlator(drone_raster)
camera = RasterCamera(drone_raster, CameraConfig(set_config.fov_w_deg))
poses = sample_route(spec.waypoints, set_config.speed_mps, set_config.step_s, set_config.altitude_m)
self_ins = InsConfig(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, SELF_MATCH_INITIAL_ERROR_M)   # exact reported pose (amendment 2)
self_match, self_fixes = {}, {}
for name, estimator in make_filters(self_correlator).items():
    _, fixes = fly(poses, camera, InsSimulator(self_ins, spec.seed), estimator, set_config.correction_interval_s,
                   SELF_MATCH_INITIAL_ERROR_M)
    self_fixes[name] = fixes
    self_match[name] = {"error_after_fix_m": fixes["error_after_m"].iloc[SELF_MATCH_FIX - 1].round(2),
                        "median_fix_ms": float(fixes["fix_ms"].median())}
checks["self_match_converges_exact_pose"] = {**self_match, "pass": all(v["error_after_fix_m"] <= CONVERGED_M for v in self_match.values())}
checks["timing"] = {name: v["median_fix_ms"] for name, v in self_match.items()}
checks["timing"]["pass"] = all(v["median_fix_ms"] <= MAX_FIX_MS for v in self_match.values())
checks["self_match_converges_exact_pose"], checks["timing"]

In [ ]:
size_px = int(SYNTHETIC_SIZE_M / SYNTHETIC_MPP)
world_center = LatLon(49.0, 25.0)
perfect_ins = InsConfig(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0)
flat_colours, flat_probabilities = class_world(np.zeros((size_px, size_px), np.uint8), world_center, SYNTHETIC_MPP)
flat_poses = straight_poses(offset_latlon(world_center, 0.0, -100.0), 0.0, 200.0)
flat = {}
for name, estimator in make_filters(synthetic_correlator(flat_probabilities)).items():
    track, fixes = fly(flat_poses, RasterCamera(flat_colours, CameraConfig()), InsSimulator(perfect_ins, SEED), estimator,
                       FIX_INTERVAL_S, 1.0)
    trace_m2 = float(track[["cov_ee", "cov_nn"]].iloc[-1].sum())
    expected_m2 = 2 * (1.0 + estimator.config.q_pos_m_per_sqrt_s ** 2 * track["t_s"].iloc[-1])
    flat[name] = {"all_blind": bool((fixes["decision"] == "blind").all()),
                  "end_offset_from_ins_m": float(track["est_error_m"].iloc[-1] - track["ins_error_m"].iloc[-1]),
                  "cov_trace_ratio": trace_m2 / expected_m2}
checks["flat_world_follows_ins"] = {**flat, "pass": all(v["all_blind"] and abs(v["end_offset_from_ins_m"]) < 0.5
                                                        and 0.8 < v["cov_trace_ratio"] < 1.2 for v in flat.values())}
checks["flat_world_follows_ins"]

In [ ]:
road_ids = np.zeros((size_px, size_px), np.uint8)
paint(road_ids, SYNTHETIC_MPP, 2, (-ROAD_WIDTH_M / 2, ROAD_WIDTH_M / 2), (-SYNTHETIC_SIZE_M / 2, SYNTHETIC_SIZE_M / 2))
road_colours, road_probabilities = class_world(road_ids, world_center, SYNTHETIC_MPP)
road_poses = straight_poses(offset_latlon(world_center, 0.0, -60.0), 0.0, 120.0)
road = {}
for name, estimator in make_filters(synthetic_correlator(road_probabilities)).items():
    estimator.start(road_poses[0].position, 5.0)
    frame = SimulatedFrames().frame(RasterCamera(road_colours, CameraConfig()).view(road_poses[0]), road_poses[0].position,
                                    0.0, ALTITUDE_M)
    record = estimator.update(frame)
    ratio, bearing = long_axis(estimator.estimate()[1])
    road[name] = {"decision": record.decision, "s1_over_s2": round(ratio, 2), "long_axis_bearing_deg": round(bearing, 1)}
checks["straight_road_elongates"] = {**road, "pass": all(v["s1_over_s2"] > ELONGATION_MIN
                                                         and min(v["long_axis_bearing_deg"], 180 - v["long_axis_bearing_deg"]) < 20
                                                         for v in road.values())}
checks["straight_road_elongates"]

In [ ]:
twin_ids = np.zeros((size_px, size_px), np.uint8)
half = BLOCK_SIZE_M / 2
for east in (0.0, BLOCK_SEPARATION_M):
    paint(twin_ids, SYNTHETIC_MPP, 1, (east - half, east + half), (-half, half))
paint(twin_ids, SYNTHETIC_MPP, 2, (-20.0, 20.0), (DISTINGUISH_ROAD_NORTH_M - 3.0, DISTINGUISH_ROAD_NORTH_M + 3.0))
twin_colours, twin_probabilities = class_world(twin_ids, world_center, SYNTHETIC_MPP)
twin_camera, twin_frames = RasterCamera(twin_colours, CameraConfig()), SimulatedFrames()
tracker = HypothesisTracker(synthetic_correlator(twin_probabilities), HypothesisTrackerConfig())
tracker.start(offset_latlon(world_center, BLOCK_SEPARATION_M / 2, 0.0), 25.0)
truth, history = world_center, []
for fix in range(3):
    prior, _ = tracker.estimate()
    record = tracker.update(twin_frames.frame(twin_camera.view(Pose(0.0, truth, 0.0, ALTITUDE_M)), prior, 0.0, ALTITUDE_M))
    weights = np.exp(tracker.log_weights)
    east = tracker.means[:, 0] + east_north_m(world_center, tracker.origin)[0]
    near_true, near_twin = weights[np.abs(east) < 10].sum(), weights[np.abs(east - BLOCK_SEPARATION_M) < 10].sum()
    history.append({"fix": fix + 1, "decision": record.decision, "weight_true": near_true, "weight_twin": near_twin,
                    "live": len(weights)})
    steps = int(FIX_INTERVAL_S / STEP_S)
    for _ in range(steps):   # north, so the next fix sees the road that only the true block has
        tracker.predict((0.0, DISTINGUISH_ROAD_NORTH_M / steps), STEP_S)
        truth = offset_latlon(truth, 0.0, DISTINGUISH_ROAD_NORTH_M / steps)
twins = pd.DataFrame(history)
first = twins.iloc[0]
checks["twin_peaks_kept_then_pruned"] = {
    "first_fix_ratio": float(min(first.weight_true, first.weight_twin) / max(first.weight_true, first.weight_twin)),
    "twin_weight_after_2_more_fixes": float(twins["weight_twin"].iloc[2]),
    "pass": bool(min(first.weight_true, first.weight_twin) / max(first.weight_true, first.weight_twin) >= TWIN_WEIGHT_RATIO_MIN
                 and twins["weight_twin"].iloc[2] < PRUNED_WEIGHT_MAX)}
twins

# Plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for name, fixes in self_fixes.items():
    axes[0].plot(fixes["t_s"], fixes["error_after_m"], marker="o", label=name)
axes[0].axhline(CONVERGED_M, color="grey", ls="--")
axes[0].set(xlabel="t, s", ylabel="error after fix, m", title="self-match, exact pose, 10 m initial error", yscale="log")
axes[0].legend()
axes[1].bar(twins["fix"] - 0.15, twins["weight_true"], 0.3, label="near true block")
axes[1].bar(twins["fix"] + 0.15, twins["weight_twin"], 0.3, label="near twin block")
axes[1].set(xlabel="fix", ylabel="hypothesis weight", title="two identical blocks, then a road")
axes[1].legend()
fig.tight_layout()

In [ ]:
pd.Series({name: value["pass"] for name, value in checks.items()})

In [ ]:
artifacts = REPO_ROOT / "experiments/027-sequence-filters/artifacts"
artifacts.mkdir(exist_ok=True)
(artifacts / "checks.json").write_text(json.dumps(checks, indent=1, default=float))
fig.savefig(REPO_ROOT / "experiments/027-sequence-filters/figures/checks.png", dpi=110)